# RAILGUN+ — Full Evaluation (Kaggle)



## 1. Setup

In [ ]:
import os, sys, pickle
!git clone -q https://github.com/tay805/railgun-plus.git /kaggle/working/railgun-plus
!pip install -q pogema
sys.path.insert(0,'/kaggle/working/railgun-plus/src')
import torch

# EDIT: paths to your checkpoint AND your LaCAM binary (both as Kaggle datasets)
CKPT='/kaggle/input/your-checkpoints/best.pt'
LACAM_BIN='/kaggle/input/lacam-binary/main'   # set to None if you don't have LaCAM

if LACAM_BIN and os.path.exists(LACAM_BIN):
    !chmod +x {LACAM_BIN}
    print('using LaCAM expert:', LACAM_BIN)
else:
    LACAM_BIN = None
    print('no LaCAM binary -> expert column will run PIBT (expert==pibt_only)')

RESULTS_DIR='/kaggle/working/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Load model (best.pt)

In [ ]:
from railgun_plus.models import RailgunUNet
device='cuda' if torch.cuda.is_available() else 'cpu'
model=RailgunUNet(6,5,base=64).to(device)
ck=torch.load(CKPT,map_location=device)
model.load_state_dict(ck['model']);model.eval()
print('loaded epoch',ck.get('epoch'),'on',device)

## 3. Build held-out test sets (fixed, saved, resumable)

In [ ]:
from railgun_plus.data.generate import generate_with_pogema
AGENT_COUNTS=[16,32,64,96,128]
N_PER=50
TEST_PATH=f'{RESULTS_DIR}/test_sets.pkl'
test_sets={}
if os.path.exists(TEST_PATH):
    test_sets=pickle.load(open(TEST_PATH,'rb'))
for k in AGENT_COUNTS:
    if test_sets.get(k):
        print(k,'agents: have',len(test_sets[k]),'(skip)'); continue
    test_sets[k]=generate_with_pogema(N_PER,32,0.2,k,seed=1000+k)
    print(k,'agents:',len(test_sets[k]),'generated')
    pickle.dump(test_sets,open(TEST_PATH,'wb'))
print('test sets:',{k:len(v) for k,v in test_sets.items()})

## 4. Run the full sweep (all four methods)
With LACAM_BIN set, the `expert` column runs LaCAM and should beat `pibt_only`.

In [ ]:
from railgun_plus.eval.harness import run_sweep, sweep_to_table, plot_sweep
sweep=run_sweep(model,test_sets,device=device,lacam_bin=LACAM_BIN)

## 5. Results tables (CSR / deadlock-rate / SoC-ratio / makespan)

In [ ]:
import pandas as pd
rows=sweep_to_table(sweep,out_csv=f'{RESULTS_DIR}/results_table.csv')
df=pd.DataFrame(rows)
print('=== CSR (higher=better) ==='); display(df.pivot(index='agents',columns='method',values='csr'))
print('=== Deadlock rate (lower=better) ==='); display(df.pivot(index='agents',columns='method',values='deadlock_rate'))
print('=== SoC ratio (closer to 1=better; solved only) ==='); display(df.pivot(index='agents',columns='method',values='avg_soc_ratio'))
df

## 6. Plots

In [ ]:
plot_sweep(sweep,metric='csr',title='CSR vs agents (higher=better)',savepath=f'{RESULTS_DIR}/csr.png')

In [ ]:
plot_sweep(sweep,metric='deadlock_rate',title='Deadlock rate vs agents (lower=better)',savepath=f'{RESULTS_DIR}/deadlock.png')

In [ ]:
plot_sweep(sweep,metric='avg_soc_ratio_solved',title='SoC ratio (1.0=optimal; solved only)',savepath=f'{RESULTS_DIR}/soc_ratio.png')

In [ ]:
plot_sweep(sweep,metric='avg_makespan_solved',title='Makespan vs agents (solved)',savepath=f'{RESULTS_DIR}/makespan.png')